In [1]:
# ============================================================================
# 1_generate_perturb.ipynb  —  Generation (gap-given/blind) + perturbations P1-P5
# ----------------------------------------------------------------------------
# MERGES former notebooks 01_generation + 02_perturbation. LLM calls.
# Run order: 0 -> 1 -> 2 -> 3 -> 4.
#
# INPUT   data/with_override.parquet            (from notebook 0)
# OUTPUT  data/generated.parquet                (case_id, condition, rating_gen, rationale)
#         data/perturbed.parquet                (case_id, condition, pert, k, rating, rationale, ...)
#
# NOTE    Requires OPENAI_API_KEY in .env. Set MAX_CASES to an int for a cheap
#         smoke-test first; None runs the full override subset (the paper's 114).
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)
import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

cfg = load_env()
from openai import OpenAI
client = OpenAI(api_key=cfg["api_key"])

GEN_MODEL = cfg["mid"]          # generator + P2/P4 editor tier
EDIT_MODEL = cfg["mid"]
MAX_CASES = None                # None = all overrides (114). Set an int to cap for a cost check.
TEMPERATURE = 0.2
MDNA_BUDGET = 6000

RATING_ORDER = ["D", "C", "CC", "CCC", "B", "BB", "BBB", "A", "AA", "AAA"]
idx = {g: i for i, g in enumerate(RATING_ORDER)}
def shift_grade(g, k):
    if g not in idx: return g
    return RATING_ORDER[max(0, min(len(RATING_ORDER) - 1, idx[g] + k))]

# ----------------------------------------------------------------------------
# PART A — generation under gap-given / gap-blind
# ----------------------------------------------------------------------------
ov = pd.read_parquet(require_input(DATA_DIR / "with_override.parquet",
                                   hint="Run notebook 0 first."))
ov = ov[ov["override"]].copy()
if MAX_CASES is not None:
    ov = ov.head(MAX_CASES)

def gen_one(row, condition):
    gap_txt = (f"The agency rating sits {int(row.delta):+d} notches versus the "
               f"quantitative stand-alone profile." if condition == "gap_given" else "")
    mdna = str(row.get("mdna_text", ""))[:MDNA_BUDGET]
    sys_msg = ("You are a credit analyst. Produce a letter rating and a short "
               "rationale citing quantitative and qualitative factors (parent/"
               "group/government support, industry, governance).")
    user_msg = (f"Financial sector: {row.get('Sector','NA')}.\n{gap_txt}\n"
                f"Relevant MD&A excerpt:\n{mdna}\n\n"
                f"Return JSON: {{\"rating\": <letter>, \"rationale\": <text>}}")
    r = client.chat.completions.create(
        model=GEN_MODEL, temperature=TEMPERATURE,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user", "content": user_msg}],
        response_format={"type": "json_object"})
    return json.loads(r.choices[0].message.content)

gen_rows = []
for condition in ("gap_given", "gap_blind"):
    for _, row in ov.iterrows():
        try:
            out = gen_one(row, condition)
        except Exception as e:
            out = {"rating": None, "rationale": f"[gen-error] {e}"}
        gen_rows.append({"case_id": f"{row.cik}_{row.rating_year}",
                         "cik": row.cik, "rating_year": int(row.rating_year),
                         "delta": int(row.delta), "condition": condition,
                         "rating_gen": out.get("rating"),
                         "rationale": out.get("rationale")})
gen = pd.DataFrame(gen_rows)
gen.to_parquet(DATA_DIR / "generated.parquet", index=False)
print("generated:", gen.groupby("condition").size().to_dict())

# ----------------------------------------------------------------------------
# PART B — controlled perturbations P1-P5 (P1 dose-response k=1,2,3)
# ----------------------------------------------------------------------------
def llm_edit(instruction, rationale):
    r = client.chat.completions.create(
        model=EDIT_MODEL, temperature=0.0,
        messages=[{"role": "system", "content":
                   "You edit a credit rationale as instructed. Return only the edited text."},
                  {"role": "user", "content": instruction + "\n\nRATIONALE:\n" + str(rationale)}])
    return r.choices[0].message.content.strip()

pert_rows = []
for _, r in gen.iterrows():
    base = {"case_id": r.case_id, "cik": r.cik, "rating_year": r.rating_year,
            "delta": r.delta, "condition": r.condition}
    g, t = r.rating_gen, r.rationale
    pert_rows.append({**base, "pert": "P0", "k": 0, "rating": g, "rationale": t})
    for k in (1, 2, 3):
        pert_rows.append({**base, "pert": "P1", "k": k, "rating": shift_grade(g, k), "rationale": t})
    pert_rows.append({**base, "pert": "P2", "k": 0, "rating": g,
                      "rationale": llm_edit("Delete every sentence that provides a "
                      "qualitative support justification (parent/group/government "
                      "support, industry, governance). Keep the rest verbatim.", t)})
    pert_rows.append({**base, "pert": "P3", "k": 0, "rating": g,
                      "rationale": llm_edit("Insert exactly one quantitative claim that "
                      "contradicts the financials (e.g. flip a ratio's direction). "
                      "Change nothing else.", t)})
    pert_rows.append({**base, "pert": "P4", "k": 0, "rating": g,
                      "rationale": llm_edit("Reverse the polarity of the qualitative "
                      "support argument (support -> drag, or drag -> support) while "
                      "keeping the same grade and surface form.", t)})
    pert_rows.append({**base, "pert": "P5", "k": 0, "rating": g,
                      "rationale": (str(t) + " " + str(t))})

pert = pd.DataFrame(pert_rows)
pert.to_parquet(DATA_DIR / "perturbed.parquet", index=False)
print("perturbed:", pert.groupby("pert").size().to_dict())
print("\n[done] generation + perturbation complete.")


generated: {'gap_blind': 114, 'gap_given': 114}
perturbed: {'P0': 228, 'P1': 684, 'P2': 228, 'P3': 228, 'P4': 228, 'P5': 228}

[done] generation + perturbation complete.
